# CareConnect on AgentCore — Lab 0: Prerequisites & Foundations

This notebook stands up the shared foundations for the **SDK / notebook** build of
CareConnect, the Riverside Health multi-agent patient-support assistant.

You have already built CareConnect once using the **Console + CLI on Ubuntu EC2**.
This second pass rebuilds the same system from **SageMaker notebooks using the AWS
SDK (boto3)** with less manual clicking.

**Key rules for this build**
- Every NEW resource is suffixed `-sdk` so nothing collides with your existing build.
- We **reuse** the existing `careconnect-approved-docs` S3 bucket (Option B) so both
  builds share the same approved Riverside Health documents. We do **not** re-upload data.
- All IDs (KB, guardrail, table, ARNs) are written to **SSM Parameter Store** so later
  notebooks read them instead of hard-coding.

> Run this notebook in the **same Region** as your existing build (us-east-1).

### Step 1: Install dependencies

In [ ]:
# Run once per kernel. Restart the kernel afterwards if prompted.
%pip install -r requirements.txt --quiet

### Step 2: Import helpers and confirm identity/Region

In [ ]:
import boto3, json, time
import lab_helpers.utils as u

print("Account:", u.get_aws_account_id())
print("Region :", u.REGION)
print("Docs bucket (reused):", u.EXISTING_DOCS_BUCKET, "/", u.DOCS_PREFIX)
print("Suffix for new resources:", u.RESOURCE_SUFFIX)
assert u.REGION == "us-east-1", "Switch your notebook Region to us-east-1 to match the existing build."

### Step 3: Confirm the reused documents bucket

Option B — we point the new Knowledge Base at the **existing** bucket. This cell only
*verifies* the approved documents are present; it does not modify them.

In [ ]:
s3 = boto3.client("s3")
resp = s3.list_objects_v2(Bucket=u.EXISTING_DOCS_BUCKET, Prefix=u.DOCS_PREFIX)
docs = [o["Key"] for o in resp.get("Contents", [])]
print(f"Found {len(docs)} objects under {u.DOCS_PREFIX} (showing first 10):")
for k in docs[:10]:
    print("  ", k)
assert docs, "No approved documents found — check the bucket name in lab_helpers/utils.py"

### Step 4: Create the Knowledge Base execution role

Least-privilege role Bedrock assumes to read the bucket and call the Titan embed model.

In [ ]:
kb_role_arn = u.create_kb_execution_role()
print("KB role:", kb_role_arn)

### Step 5: Create the Knowledge Base backed by S3 Vectors

We create a new `careconnect-kb-sdk` Knowledge Base. This mirrors the console
"Quick create → Amazon S3 Vectors" flow, done through boto3.

> Exact request shapes for S3-Vectors-backed KBs evolve with the SDK. If
> `create_knowledge_base` rejects the `storageConfiguration` below, print the error,
> check the installed botocore version, and adjust to the current
> `s3VectorsConfiguration` schema. The rest of the notebook is unaffected.

In [ ]:
bedrock_agent = boto3.client("bedrock-agent", region_name=u.REGION)
account = u.get_aws_account_id()

embed_arn = f"arn:aws:bedrock:{u.REGION}::foundation-model/{u.EMBED_MODEL_ID}"

# NOTE: S3 Vectors KB creation via SDK — adjust to your botocore version if needed.
try:
    kb = bedrock_agent.create_knowledge_base(
        name=u.KB_NAME,
        description="CareConnect approved Riverside Health documents (SDK build).",
        roleArn=kb_role_arn,
        knowledgeBaseConfiguration={
            "type": "VECTOR",
            "vectorKnowledgeBaseConfiguration": {"embeddingModelArn": embed_arn},
        },
        storageConfiguration={
            "type": "S3_VECTORS",
            "s3VectorsConfiguration": {
                # Let Bedrock create/manage the vector bucket + index for the lab.
                "vectorBucketArn": f"arn:aws:s3vectors:{u.REGION}:{account}:bucket/{u.KB_NAME}",
                "indexName": f"{u.KB_NAME}-index",
            },
        },
    )
    kb_id = kb["knowledgeBase"]["knowledgeBaseId"]
    print("Created KB:", kb_id)
except Exception as e:
    print("create_knowledge_base failed — inspect and adjust schema:\n", e)
    raise

In [ ]:
u.wait_for_kb_ready(kb_id)
u.put_ssm_parameter(f"{u.SSM_PREFIX}/kb_id", kb_id)
print("Saved KB id to SSM:", kb_id)

### Step 6: Attach the reused S3 bucket as a data source and sync

In [ ]:
ds_resp = bedrock_agent.create_data_source(
    knowledgeBaseId=kb_id,
    name=u.name("careconnect-approved-documents"),
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": f"arn:aws:s3:::{u.EXISTING_DOCS_BUCKET}",
            "inclusionPrefixes": [u.DOCS_PREFIX],
        },
    },
)
data_source_id = ds_resp["dataSource"]["dataSourceId"]
print("Data source:", data_source_id)

job = bedrock_agent.start_ingestion_job(
    knowledgeBaseId=kb_id, dataSourceId=data_source_id)
print("Ingestion job started:", job["ingestionJob"]["ingestionJobId"])

In [ ]:
# Poll the ingestion job to completion
while True:
    j = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=kb_id, dataSourceId=data_source_id,
        ingestionJobId=job["ingestionJob"]["ingestionJobId"])["ingestionJob"]
    print("ingestion:", j["status"])
    if j["status"] in ("COMPLETE", "FAILED"):
        break
    time.sleep(15)

### Step 7: Quick retrieval test

In [ ]:
rt = boto3.client("bedrock-agent-runtime", region_name=u.REGION)
r = rt.retrieve(
    knowledgeBaseId=kb_id,
    retrievalQuery={"text": "How should a patient prepare for a colonoscopy?"},
    retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}},
)
for res in r["retrievalResults"]:
    print(res["location"].get("s3Location", {}).get("uri"), "score", res.get("score"))

### Step 8: Create the CareConnect Patient Safety Guardrail

Same policy as the console build — denied topics (Diagnosis, Dosage, Treatment, Triage),
PII masking, and the custom MRN regex — created via boto3.

In [ ]:
bedrock = boto3.client("bedrock", region_name=u.REGION)

blocked_msg = ("I can help with approved Riverside Health information, but I cannot "
               "provide medical diagnosis, medication dosage changes, treatment "
               "recommendations, or medical triage. Please contact a qualified "
               "healthcare professional for medical advice.")

g = bedrock.create_guardrail(
    name=u.GUARDRAIL_NAME,
    description="Safety and privacy guardrail for CareConnect (SDK build).",
    blockedInputMessaging=blocked_msg,
    blockedOutputsMessaging=blocked_msg,
    topicPolicyConfig={"topicsConfig": [
        {"name": "Medical Diagnosis", "type": "DENY",
         "definition": "Questions or statements that identify, confirm, or infer a medical condition based on symptoms, test results, or patient information."},
        {"name": "Medication Dosage", "type": "DENY",
         "definition": "Questions, guidance, or recommendations about medication dose, changing a dose, medication frequency, or taking more or less medication than prescribed."},
        {"name": "Treatment Recommendation", "type": "DENY",
         "definition": "Questions or recommendations that choose, prescribe, or recommend a medical treatment, medication, procedure, or therapy for a patient."},
        {"name": "Medical Triage", "type": "DENY",
         "definition": "Questions that decide how urgently a patient needs care or which level of care to seek."},
    ]},
    sensitiveInformationPolicyConfig={
        "piiEntitiesConfig": [
            {"type": "EMAIL", "action": "ANONYMIZE"},
            {"type": "PHONE", "action": "ANONYMIZE"},
            {"type": "NAME", "action": "ANONYMIZE"},
            {"type": "ADDRESS", "action": "ANONYMIZE"},
        ],
        "regexesConfig": [
            {"name": "MRN", "pattern": "MRN-[0-9]{8}", "action": "ANONYMIZE",
             "description": "Synthetic Riverside Health Medical Record Number."},
        ],
    },
)
guardrail_id = g["guardrailId"]
print("Guardrail:", guardrail_id)

ver = bedrock.create_guardrail_version(guardrailIdentifier=guardrail_id,
                                       description="v1 for CareConnect SDK build")
guardrail_version = ver["version"]
u.put_ssm_parameter(f"{u.SSM_PREFIX}/guardrail_id", guardrail_id)
u.put_ssm_parameter(f"{u.SSM_PREFIX}/guardrail_version", guardrail_version)
print("Guardrail version:", guardrail_version)

## Lab 0 complete ✅

You now have, all suffixed `-sdk` and recorded in SSM:
- a Knowledge Base over the **reused** approved-docs bucket, synced and tested
- the Patient Safety Guardrail (versioned)

Next: **lab-01** builds the Retrieval and Document-Processing agents.